In [ ]:
import pandas as pd
from collections import Counter

from google.colab import files
import pandas as pd
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print("Uploaded file:", filename)
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")

display(df.head())
print("\nColumns:")
print(df.columns.tolist())


Saving mushrooms (1).csv to mushrooms (1) (1).csv
Uploaded file: mushrooms (1) (1).csv
Dataset shape: (8124, 23)

First 5 rows:


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g



Columns:
['class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring', 'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat']


In [ ]:
df = df.astype(str)

df = df.replace("?", "missing")

target = "class"

features = [

    col for col in df.columns

    if col != target

]

print("\nTarget column:", target)

print("\nFeatures:")

print(features)

print("\nClass distribution:")

print(df[target].value_counts())


Target column: class

Features:
['cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring', 'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat']

Class distribution:
class
e    4208
p    3916
Name: count, dtype: int64


In [ ]:
df = df.sample(

    frac=1,

    random_state=42

).reset_index(drop=True)

# 80% training, 20% testing

split_index = int(0.8 * len(df))

train_df = df.iloc[:split_index].copy()

test_df = df.iloc[split_index:].copy()

# Display information

print("Total records:", len(df))

print("Training records:", len(train_df))

print("Testing records:", len(test_df))

print("\nNumber of features:", len(features))

print("\nFeatures:")

for feature in features:

    print("-", feature)

print("\nClass distribution in training data:")

print(train_df[target].value_counts())

print("\nClass distribution in testing data:")

print(test_df[target].value_counts())

Total records: 8124
Training records: 6499
Testing records: 1625

Number of features: 22

Features:
- cap-shape
- cap-surface
- cap-color
- bruises
- odor
- gill-attachment
- gill-spacing
- gill-size
- gill-color
- stalk-shape
- stalk-root
- stalk-surface-above-ring
- stalk-surface-below-ring
- stalk-color-above-ring
- stalk-color-below-ring
- veil-type
- veil-color
- ring-number
- ring-type
- spore-print-color
- population
- habitat

Class distribution in training data:
class
e    3361
p    3138
Name: count, dtype: int64

Class distribution in testing data:
class
e    847
p    778
Name: count, dtype: int64


In [24]:
def covers_rule(row, rule):


    for feature, value in rule.items():

        if row[feature] != value:

            return False

    return True


def rule_coverage(data, rule):

    mask = data.apply(

        lambda row: covers_rule(row, rule),

        axis=1

    )

    return data[mask]


def rule_precision(data, rule, target_value):

    covered = rule_coverage(data, rule)

    if len(covered) == 0:

        return 0

    correct = sum(

        covered[target] == target_value

    )

    precision = correct / len(covered)

    return precision


def print_rule(rule, class_value):

    if len(rule) == 0:

        print(

            f"IF TRUE THEN class = {class_value}"

        )

        return

    conditions = []

    for feature, value in rule.items():

        conditions.append(

            f"{feature} = {value}"

        )

    rule_text = " AND ".join(conditions)

    print(

        f"IF {rule_text} THEN class = {class_value}"

    )


def predict_row(row, rules, default_class):

    for rule, rule_class in rules:

        if covers_rule(row, rule):

            return rule_class

    return default_class


def calculate_accuracy(data, rules, default_class):

    correct = 0

    for _, row in data.iterrows():

        prediction = predict_row(

            row,

            rules,

            default_class

        )

        if prediction == row[target]:

            correct += 1

    accuracy = correct / len(data)

    return accuracy

print("Helper functions created successfully!")

Helper functions created successfully!


In [25]:
def find_best_condition(data, target_value, used_features):

    best_condition = None

    best_precision = -1

    for feature in features:

        if feature in used_features:

            continue

        for value in data[feature].unique():

            candidate_rule = {

                feature: value

            }

            covered = rule_coverage(

                data,

                candidate_rule

            )

            if len(covered) == 0:

                continue

            positive = sum(

                covered[target] == target_value

            )


            precision = positive / len(covered)

            if precision > best_precision:

                best_precision = precision

                best_condition = (

                    feature,

                    value

                )

    return best_condition, best_precision


def ripper(train_data):

    print("\n")

    print("=" * 70)

    print("RIPPER RULE LEARNING")

    print("=" * 70)

    rules = []

    class_counts = train_data[target].value_counts()

    print("\nClass counts:")

    print(class_counts)


    target_class = class_counts.index[-1]

    print("\nTarget class for rule learning:", target_class)


    remaining = train_data.copy()

    rule_number = 1


    while len(remaining) > 0:

        positive_examples = remaining[

            remaining[target] == target_class

        ]


        if len(positive_examples) == 0:

            break


        rule = {}


        used_features = set()

        while True:

            condition, precision = find_best_condition(

                remaining,

                target_class,

                used_features

            )


            if condition is None:

                break

            feature, value = condition


            rule[feature] = value

            used_features.add(feature)

            covered = rule_coverage(

                remaining,

                rule

            )

            if len(covered) == 0:

                break

            positive = sum(

                covered[target] == target_class

            )

            precision = positive / len(covered)


            if precision >= 0.95:

                break

        covered = rule_coverage(

            remaining,

            rule

        )

        if len(covered) == 0:

            break

        positive = sum(

            covered[target] == target_class

        )

        precision = positive / len(covered)

        if precision >= 0.80:

            rules.append(

                (rule, target_class)

            )

            print(

                f"\nRule {rule_number}:"

            )

            print_rule(

                rule,

                target_class

            )

            print(

                "Covered examples:",

                len(covered)

            )

            print(

                "Precision:",

                round(precision * 100, 2),

                "%"

            )

            mask = remaining.apply(

                lambda row: covers_rule(

                    row,

                    rule

                ),

                axis=1

            )

            remaining = remaining[~mask]

            rule_number += 1

        else:

            break

    default_class = (

        train_data[target]

        .value_counts()

        .index[0]

    )

    print("\nDefault class:", default_class)

    print(

        "\nTotal RIPPER rules generated:",

        len(rules)

    )

    return rules, default_class

ripper_rules, ripper_default =ripper(

    train_df

)



RIPPER RULE LEARNING

Class counts:
class
e    3361
p    3138
Name: count, dtype: int64

Target class for rule learning: p

Rule 1:
IF cap-shape = c THEN class = p
Covered examples: 3
Precision: 100.0 %

Rule 2:
IF cap-surface = g THEN class = p
Covered examples: 3
Precision: 100.0 %

Rule 3:
IF odor = y THEN class = p
Covered examples: 459
Precision: 100.0 %

Rule 4:
IF odor = f THEN class = p
Covered examples: 1731
Precision: 100.0 %

Rule 5:
IF odor = s THEN class = p
Covered examples: 462
Precision: 100.0 %

Rule 6:
IF odor = p THEN class = p
Covered examples: 206
Precision: 100.0 %

Rule 7:
IF odor = c THEN class = p
Covered examples: 156
Precision: 100.0 %

Rule 8:
IF odor = m THEN class = p
Covered examples: 26
Precision: 100.0 %

Rule 9:
IF gill-color = r THEN class = p
Covered examples: 19
Precision: 100.0 %

Rule 10:
IF stalk-color-above-ring = y THEN class = p
Covered examples: 6
Precision: 100.0 %

Rule 11:
IF stalk-color-below-ring = y THEN class = p
Covered examples: 12

In [26]:
print("=" * 70)

print("RIPPER EVALUATION")

print("=" * 70)

# Calculate accuracy on test data

ripper_accuracy = calculate_accuracy(

    test_df,

    ripper_rules,

    ripper_default

)

print(

    "\nRIPPER Test Accuracy:",

    round(ripper_accuracy * 100, 2),

    "%"

)

RIPPER EVALUATION

RIPPER Test Accuracy: 100.0 %


In [27]:
print("\nSample Predictions:")

print("-" * 70)


for index, (_, row) in enumerate(

    test_df.head(10).iterrows(),

    start=1

):

    actual = row[target]

    predicted = predict_row(

        row,

        ripper_rules,

        ripper_default

    )

    print(

        f"{index}. Actual = {actual} | "

        f"Predicted = {predicted}"

    )


Sample Predictions:
----------------------------------------------------------------------
1. Actual = e | Predicted = e
2. Actual = p | Predicted = p
3. Actual = p | Predicted = p
4. Actual = p | Predicted = p
5. Actual = p | Predicted = p
6. Actual = p | Predicted = p
7. Actual = p | Predicted = p
8. Actual = p | Predicted = p
9. Actual = p | Predicted = p
10. Actual = p | Predicted = p


In [28]:
def cn2(train_data):

    print("\n")

    print("=" * 70)

    print("CN2 RULE LEARNING")

    print("=" * 70)

    rules = []

    # Copy training data

    remaining = train_data.copy()

    rule_number = 1

    while len(remaining) >= 10:

        best_rule = None

        best_class = None

        best_score = -1

        for feature in features:

            for value in remaining[feature].unique():

                # Candidate rule

                rule = {

                    feature: value

                }

                # Find examples covered by this rule

                covered = rule_coverage(

                    remaining,

                    rule

                )

                if len(covered) == 0:

                    continue

                # Count classes among covered examples

                class_counts = Counter(

                    covered[target]

                )

                # Most common class

                rule_class = (

                    class_counts

                    .most_common(1)[0][0]

                )

                # Number of correctly classified examples

                correct = class_counts[rule_class]

                # Accuracy of rule

                accuracy = correct / len(covered)

                score = accuracy * len(covered)

                if score > best_score:

                    best_score = score

                    best_rule = rule

                    best_class = rule_class

        if best_rule is None:

            break

        rules.append(

            (best_rule, best_class)

        )

        print(

            f"\nRule {rule_number}:"

        )

        print_rule(

            best_rule,

            best_class

        )

        covered = rule_coverage(

            remaining,

            best_rule

        )

        accuracy = sum(

            covered[target] == best_class

        ) / len(covered)

        print(

            "Covered examples:",

            len(covered)

        )

        print(

            "Rule accuracy:",

            round(accuracy * 100, 2),

            "%"

        )

        mask = remaining.apply(

            lambda row: covers_rule(

                row,

                best_rule

            ),

            axis=1

        )

        remaining = remaining[~mask]

        rule_number += 1

    default_class = (

        train_data[target]

        .value_counts()

        .index[0]

    )

    print(

        "\nDefault class:",

        default_class

    )

    print(

        "\nTotal CN2 rules generated:",

        len(rules)

    )

    return rules, default_class

cn2_rules, cn2_default = cn2(

    train_df

)



CN2 RULE LEARNING

Rule 1:
IF veil-type = p THEN class = e
Covered examples: 6499
Rule accuracy: 51.72 %

Default class: e

Total CN2 rules generated: 1


In [29]:


print("=" * 70)

print("CN2 EVALUATION")

print("=" * 70)

cn2_accuracy = calculate_accuracy(

    test_df,

    cn2_rules,

    cn2_default

)

print(

    "\nCN2 Test Accuracy:",

    round(cn2_accuracy * 100, 2),

    "%"

)



CN2 EVALUATION

CN2 Test Accuracy: 52.12 %


In [30]:
print("\nSample Predictions:")

print("-" * 70)

for index, (_, row) in enumerate(

    test_df.head(10).iterrows(),

    start=1

):

    actual = row[target]

    predicted = predict_row(

        row,

        cn2_rules,

        cn2_default

    )

    print(

        f"{index}. Actual = {actual} | "

        f"Predicted = {predicted}"

    )


print("\n")

print("=" * 70)

print("RIPPER vs CN2")

print("=" * 70)

print(

    f"RIPPER Accuracy : {ripper_accuracy * 100:.2f}%"

)

print(

    f"CN2 Accuracy    : {cn2_accuracy * 100:.2f}%"

)


Sample Predictions:
----------------------------------------------------------------------
1. Actual = e | Predicted = e
2. Actual = p | Predicted = e
3. Actual = p | Predicted = e
4. Actual = p | Predicted = e
5. Actual = p | Predicted = e
6. Actual = p | Predicted = e
7. Actual = p | Predicted = e
8. Actual = p | Predicted = e
9. Actual = p | Predicted = e
10. Actual = p | Predicted = e


RIPPER vs CN2
RIPPER Accuracy : 100.00%
CN2 Accuracy    : 52.12%


In [31]:
import math

def foil_gain(

    current_data,

    new_data,

    target_value

):

    """

    Calculate FOIL Information Gain.

    p0 = positive examples before adding condition

    n0 = negative examples before adding condition

    p1 = positive examples after adding condition

    n1 = negative examples after adding condition

    """

    p0 = sum(

        current_data[target] == target_value

    )

    n0 = len(current_data) - p0

    p1 = sum(

        new_data[target] == target_value

    )

    n1 = len(new_data) - p1


    if p1 == 0:

        return -1

    # Probability before condition

    old_probability = p0 / (p0 + n0)

    # Probability after condition

    new_probability = p1 / (p1 + n1)

    # Avoid division by zero

    if old_probability == 0:

        return -1

    # FOIL Information Gain

    gain = p1 * math.log2(

        new_probability / old_probability

    )

    return gain

example_rule = {

    "odor": "f"

}


new_data = rule_coverage(

    train_df,

    example_rule

)

gain = foil_gain(

    train_df,

    new_data,

    "p"

)

print("Candidate condition:")

print_rule(

    example_rule,

    "p"

)

print(

    "\nFOIL Information Gain:",

    round(gain, 4)

)

Candidate condition:
IF odor = f THEN class = p

FOIL Information Gain: 1818.1946


In [32]:

def find_best_foil_condition(
    current_data,
    target_value,
    used_features
):

    best_condition = None
    best_gain = -999999

    # Try every feature
    for feature in features:

        # Don't reuse same feature
        if feature in used_features:
            continue

        # Try every possible value
        for value in current_data[feature].unique():

            # Candidate condition
            candidate_rule = {
                feature: value
            }

            # Examples covered by condition
            new_data = rule_coverage(
                current_data,
                candidate_rule
            )

            if len(new_data) == 0:
                continue

            # Calculate FOIL gain
            gain = foil_gain(
                current_data,
                new_data,
                target_value
            )

            # Keep highest gain
            if gain > best_gain:

                best_gain = gain

                best_condition = (
                    feature,
                    value
                )

    return best_condition, best_gain


target_class = train_df[target].value_counts().index[-1]

condition, gain = find_best_foil_condition(
    train_df,
    target_class,
    set()
)

print("Target class:", target_class)

print(
    "\nBest first condition:"
)

print(
    condition
)

print(
    "\nFOIL Gain:",
    round(gain, 4)
)


Target class: p

Best first condition:
('odor', 'f')

FOIL Gain: 1818.1946


In [33]:
def foil(train_data):

    print("\n")

    print("=" * 70)

    print("FOIL - FIRST ORDER INDUCTIVE LEARNER")

    print("=" * 70)

    rules = []

    class_counts = train_data[target].value_counts()

    # Learn rules for minority class

    target_class = class_counts.index[-1]

    print(

        "\nTarget class:",

        target_class

    )

    # Copy training data

    remaining = train_data.copy()

    rule_number = 1

    while True:

        # Positive examples still not covered

        positives = remaining[

            remaining[target] == target_class

        ]

        # Stop if no positives remain

        if len(positives) == 0:

            break

        # Start with empty rule

        rule = {}

        # Data currently covered by rule

        current_data = remaining.copy()

        # Features already used

        used_features = set()

        while True:

            # Find condition with maximum FOIL gain

            condition, gain = find_best_foil_condition(

                current_data,

                target_class,

                used_features

            )

            # No condition found

            if condition is None:

                break

            # No improvement

            if gain <= 0:

                break

            feature, value = condition

            # Add condition to rule

            rule[feature] = value

            used_features.add(feature)


            current_data = rule_coverage(

                current_data,

                {

                    feature: value

                }

            )

            negative_count = sum(

                current_data[target] != target_class

            )

            if negative_count == 0:

                break

        if len(rule) == 0:

            break

        covered = rule_coverage(

            remaining,

            rule

        )

        positive_count = sum(

            covered[target] == target_class

        )

        negative_count = (

            len(covered) - positive_count

        )

        if positive_count == 0:

            break


        rules.append(

            (rule, target_class)

        )

        print(

            f"\nRule {rule_number}:"

        )

        print_rule(

            rule,

            target_class

        )

        print(

            "Covered examples:",

            len(covered)

        )

        print(

            "Positive examples:",

            positive_count

        )

        print(

            "Negative examples:",

            negative_count

        )

        mask = remaining.apply(

            lambda row: covers_rule(

                row,

                rule

            ),

            axis=1

        )

        # Remove only positive examples

        remove_positive = (

            mask

            & (remaining[target] == target_class)

        )

        remaining = remaining[

            ~remove_positive

        ]

        rule_number += 1

    default_class = (

        train_data[target]

        .value_counts()

        .index[0]

    )

    print(

        "\nDefault class:",

        default_class

    )

    print(

        "\nTotal FOIL rules generated:",

        len(rules)

    )

    return rules, default_class

foil_rules, foil_default = foil(

    train_df

)



FOIL - FIRST ORDER INDUCTIVE LEARNER

Target class: p

Rule 1:
IF odor = f THEN class = p
Covered examples: 1731
Positive examples: 1731
Negative examples: 0

Rule 2:
IF gill-size = n AND gill-color = b THEN class = p
Covered examples: 921
Positive examples: 921
Negative examples: 0

Rule 3:
IF gill-size = n AND odor = p THEN class = p
Covered examples: 206
Positive examples: 206
Negative examples: 0

Rule 4:
IF odor = c THEN class = p
Covered examples: 156
Positive examples: 156
Negative examples: 0

Rule 5:
IF spore-print-color = r THEN class = p
Covered examples: 59
Positive examples: 59
Negative examples: 0

Rule 6:
IF stalk-surface-below-ring = y AND stalk-surface-above-ring = k THEN class = p
Covered examples: 52
Positive examples: 52
Negative examples: 0

Rule 7:
IF stalk-color-above-ring = y THEN class = p
Covered examples: 8
Positive examples: 8
Negative examples: 0

Rule 8:
IF cap-surface = g THEN class = p
Covered examples: 3
Positive examples: 3
Negative examples: 0

Rule

In [34]:
print("=" * 70)

print("FOIL EVALUATION")

print("=" * 70)

# Calculate FOIL accuracy

foil_accuracy = calculate_accuracy(

    test_df,

    foil_rules,

    foil_default

)

print(

    "\nFOIL Test Accuracy:",

    round(foil_accuracy * 100, 2),

    "%"

)

print("\nSample Predictions:")

print("-" * 70)

for index, (_, row) in enumerate(

    test_df.head(10).iterrows(),

    start=1

):

    actual = row[target]

    predicted = predict_row(

        row,

        foil_rules,

        foil_default

    )

    print(

        f"{index}. Actual = {actual} | "

        f"Predicted = {predicted}"

    )


print("\n")

print("=" * 70)

print("FINAL ALGORITHM COMPARISON")

print("=" * 70)

print(

    f"{'Algorithm':<15}"

    f"{'Rules':<15}"

    f"{'Accuracy':<15}"

)

print("-" * 45)

print(

    f"{'RIPPER':<15}"

    f"{len(ripper_rules):<15}"

    f"{ripper_accuracy * 100:.2f}%"

)

print(

    f"{'CN2':<15}"

    f"{len(cn2_rules):<15}"

    f"{cn2_accuracy * 100:.2f}%"

)

print(

    f"{'FOIL':<15}"

    f"{len(foil_rules):<15}"

    f"{foil_accuracy * 100:.2f}%"

)

FOIL EVALUATION

FOIL Test Accuracy: 100.0 %

Sample Predictions:
----------------------------------------------------------------------
1. Actual = e | Predicted = e
2. Actual = p | Predicted = p
3. Actual = p | Predicted = p
4. Actual = p | Predicted = p
5. Actual = p | Predicted = p
6. Actual = p | Predicted = p
7. Actual = p | Predicted = p
8. Actual = p | Predicted = p
9. Actual = p | Predicted = p
10. Actual = p | Predicted = p


FINAL ALGORITHM COMPARISON
Algorithm      Rules          Accuracy       
---------------------------------------------
RIPPER         14             100.00%
CN2            1              52.12%
FOIL           10             100.00%
